# Phase 1 — Parse-level cleaning

This notebook applies all required parse-time cleaning steps before any EDA.


In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

raw_path = "dcs_student_data.csv"
df = pd.read_csv(raw_path)

print(f"Raw shape: {df.shape}")
df.head()


In [ ]:
score_columns = [
    "Attendance (%)",
    "Midterm_Score",
    "Final_Score",
    "Assignments_Avg",
    "Quizzes_Avg",
    "Participation_Score",
    "Projects_Score",
    "Total_Score",
    "math_score",
    "reading_score",
    "writing_score",
    "science_score",
]

# Coerce score columns and Age to numeric (non-numeric values become NaN)
for col in score_columns + ["Age"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Normalize categorical text and merge synonyms
df["Gender"] = df["Gender"].astype(str).str.strip().str.lower().map({
    "female": "Female",
    "male": "Male",
}).fillna(df["Gender"].astype(str).str.strip().str.title())

dept_map = {
    "engineering": "Engineering",
    "business": "Business",
    "mathematics": "Mathematics",
    "math": "Mathematics",
    "computer science": "Computer Science",
    "cs": "Computer Science",
}
df["Department"] = (
    df["Department"].astype(str).str.strip().str.lower().map(dept_map)
    .fillna(df["Department"].astype(str).str.strip().str.title())
)

# Apply invalid-value rules
for col in score_columns:
    df.loc[(df[col] < 0) | (df[col] > 100), col] = np.nan

df.loc[(df["Age"] < 15) | (df["Age"] > 60), "Age"] = np.nan

# Drop duplicate Student_ID rows, keep first occurrence
rows_before = len(df)
df = df.drop_duplicates(subset=["Student_ID"], keep="first").copy()
rows_after = len(df)

print(f"Rows before duplicate removal: {rows_before}")
print(f"Rows after duplicate removal:  {rows_after}")
print(f"Duplicates removed:            {rows_before - rows_after}")


In [ ]:
missing_summary = df[["Age"] + score_columns].isna().sum().sort_values(ascending=False)
print("Missing values after cleaning:")
print(missing_summary)

print("\nNormalized category values:")
print("Gender:", sorted(df["Gender"].dropna().unique().tolist()))
print("Department:", sorted(df["Department"].dropna().unique().tolist()))


In [ ]:
clean_path = "dcs_student_data_cleaned.csv"
df.to_csv(clean_path, index=False)
print(f"Saved cleaned dataset to: {clean_path}")


Phase 1 is complete: numeric coercion, categorical normalization, invalid-value handling, and duplicate removal have been applied.
